# 4-4 グラフ作成

4-3 で 5 つの集計結果が手に入りました:

- `by_date` — 日次の売上推移
- `by_region` — 地域別 (売上・利益・件数)
- `by_pref` — 県別 ランキング
- `by_product` — 商品別
- `pivot_region_cat` — 地域 × カテゴリ

これらを **3 章で習った matplotlib スタイル** で図にして、PNG ファイルとして保存します。**新しいことはほぼ何もありません** — 3 章でやったやつを順番に並べるだけ。

## このノートのゴール

4-5 で Excel レポートに貼り付ける **画像ファイル一式** を `output/charts/` に出力する。

- `daily_sales.png` — 日次売上推移
- `by_region.png` — 地域別 売上
- `by_pref_top10.png` — 県別 売上 TOP10
- `by_product.png` — 商品別 売上
- `region_category_stacked.png` — 地域 × カテゴリ 積み上げ

**3-4 で覚えた仕上げの作法** をすべて適用 (japanize / 桁区切り / 単位を軸の上に / tight_layout / savefig)。

## 準備 — 4-3 までを 1 セルに

In [ ]:
!pip install -q japanize-matplotlib

# === Colab ===
# from google.colab import drive
# drive.mount('/content/drive')
# BASE = "/content/drive/MyDrive/python-data-basics"

# === ローカル ===
BASE = "."

DATA = f"{BASE}/data/capstone"
SALES_DIR = f"{DATA}/sales_2026-01"
OUT = f"{BASE}/output"
CHARTS = f"{OUT}/charts"

import os
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import japanize_matplotlib
from pathlib import Path

os.makedirs(CHARTS, exist_ok=True)

# --- 4-2 ファイル読み込み ---
files = sorted(Path(SALES_DIR).glob("sales_2026-01_*.xlsx"))
dfs = []
for f in files:
    d = pd.read_excel(f, sheet_name="売上")
    d["prefecture_code"] = int(f.stem.split("_")[2])
    dfs.append(d)
all_sales = pd.concat(dfs, ignore_index=True)

# --- 4-3 マスタ結合 + 利益 ---
master_products = pd.read_excel(f"{DATA}/master_products.xlsx")
master_branches = pd.read_excel(f"{DATA}/master_branches.xlsx")
df = (
    all_sales
    .merge(master_products[["product_code", "category", "cost"]], on="product_code", how="left")
    .merge(master_branches, on="prefecture_code", how="left")
)
df["profit"] = df["amount"] - df["cost"] * df["quantity"]

# --- 4-3 で作った集計たち ---
by_date    = df.groupby("date")["amount"].sum()
by_region  = df.groupby("region").agg(売上=("amount","sum"), 利益=("profit","sum")).sort_values("売上", ascending=False)
by_pref    = df.groupby("prefecture_name").agg(売上=("amount","sum"), 利益=("profit","sum")).sort_values("売上", ascending=False)
by_product = df.groupby("product_name")["amount"].sum().sort_values(ascending=False)
pivot_region_cat = df.pivot_table(index="region", columns="category", values="amount", aggfunc="sum", fill_value=0)

print(f"df: {len(df):,} 行")
print(f"出力先: {CHARTS}")

## グラフ 1: 日次売上推移

3-3 で覚えた **`Series.plot()`** + 3-4 の **「単位は軸の上に」** スタイル。

In [ ]:
ax = by_date.plot(figsize=(12, 4), marker="o", color="steelblue")
ax.yaxis.set_major_formatter(
    ticker.FuncFormatter(lambda x, _: f"{x/10000:,.0f}")
)
plt.title("全社 日次売上推移 (2026年1月)", fontsize=14, fontweight="bold")
ax.text(0, 1.02, "(万円)", transform=ax.transAxes, ha="left", fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()

plt.savefig(f"{CHARTS}/daily_sales.png", dpi=150, bbox_inches="tight")
plt.show()

## グラフ 2: 地域別 売上

In [ ]:
ax = by_region["売上"].plot.bar(figsize=(9, 4), color="steelblue")
ax.yaxis.set_major_formatter(
    ticker.FuncFormatter(lambda x, _: f"{x/10000:,.0f}")
)
plt.title("地域ブロック別 売上", fontsize=14, fontweight="bold")
ax.text(0, 1.02, "(万円)", transform=ax.transAxes, ha="left", fontsize=10)
plt.xticks(rotation=0)
plt.tight_layout()

plt.savefig(f"{CHARTS}/by_region.png", dpi=150, bbox_inches="tight")
plt.show()

## グラフ 3: 県別 売上 TOP10 (横棒)

**ランキングは横棒** が読みやすい (3-3 で覚えた `barh`)。

In [ ]:
top10 = by_pref["売上"].head(10)
ax = top10[::-1].plot.barh(figsize=(8, 5), color="steelblue")
ax.xaxis.set_major_formatter(
    ticker.FuncFormatter(lambda x, _: f"{x/10000:,.0f}")
)
plt.title("県別 売上 TOP10", fontsize=14, fontweight="bold")
ax.text(0, 1.02, "(万円)", transform=ax.transAxes, ha="left", fontsize=10)
plt.tight_layout()

plt.savefig(f"{CHARTS}/by_pref_top10.png", dpi=150, bbox_inches="tight")
plt.show()

> 💡 `top10[::-1]` で **順序を反転** しています。横棒グラフは下から積まれるので、反転しないと TOP1 が一番下に来てしまう。

## グラフ 4: 商品別 売上

In [ ]:
ax = by_product.plot.bar(figsize=(9, 4), color="steelblue")
ax.yaxis.set_major_formatter(
    ticker.FuncFormatter(lambda x, _: f"{x/10000:,.0f}")
)
plt.title("商品別 売上", fontsize=14, fontweight="bold")
ax.text(0, 1.02, "(万円)", transform=ax.transAxes, ha="left", fontsize=10)
plt.xticks(rotation=20)
plt.tight_layout()

plt.savefig(f"{CHARTS}/by_product.png", dpi=150, bbox_inches="tight")
plt.show()

## グラフ 5: 地域 × カテゴリ 積み上げ棒

3-3 で覚えた **`pivot.plot.bar(stacked=True)`** がそのまま使えます。

In [ ]:
ax = pivot_region_cat.plot.bar(
    stacked=True, figsize=(10, 5), colormap="Pastel1",
)
ax.yaxis.set_major_formatter(
    ticker.FuncFormatter(lambda x, _: f"{x/10000:,.0f}")
)
plt.title("地域 × カテゴリ 売上内訳", fontsize=14, fontweight="bold")
ax.text(0, 1.02, "(万円)", transform=ax.transAxes, ha="left", fontsize=10)
plt.xticks(rotation=0)
plt.legend(loc="upper left", bbox_to_anchor=(1, 1))
plt.tight_layout()

plt.savefig(f"{CHARTS}/region_category_stacked.png", dpi=150, bbox_inches="tight")
plt.show()

## 確認 — 5 枚揃ったか

`output/charts/` フォルダの中身を確認します。

In [ ]:
for f in sorted(Path(CHARTS).glob("*.png")):
    size_kb = f.stat().st_size / 1024
    print(f"  {f.name:40s} {size_kb:6.1f} KB")

## 練習問題

本文のコードを少しだけ変える形の演習です。

1. **県別 利益 TOP10** の横棒グラフを描いて、`output/charts/by_pref_profit_top10.png` として保存してください。本文 cell-9 の **`by_pref["売上"]` を `by_pref["利益"]` に変える** だけ。タイトルも合わせて変更。
2. **地域別 件数** を縦棒グラフで描いてください。本文 cell-7 で `by_region["売上"]` の代わりに、**`df.groupby("region").size()`** を使う。
3. **`pivot_region_cat` の積み上げ** を、`colormap="Set2"` に変えて描き、`output/charts/region_category_set2.png` として保存してください。本文 cell-14 の **`colormap="Pastel1"` を `"Set2"` に変える** だけ。

In [ ]:
# ここに書いてみてください


<details>
<summary>答えを見る</summary>

```python
# 1. 県別 利益 TOP10 横棒
top10_profit = by_pref["利益"].head(10)
ax = top10_profit[::-1].plot.barh(figsize=(8, 5), color="steelblue")
ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f"{x/10000:,.0f}"))
plt.title("県別 利益 TOP10", fontsize=14, fontweight="bold")
ax.text(0, 1.02, "(万円)", transform=ax.transAxes, ha="left", fontsize=10)
plt.tight_layout()
plt.savefig(f"{CHARTS}/by_pref_profit_top10.png", dpi=150, bbox_inches="tight")
plt.show()

# 2. 地域別 件数
ax = df.groupby("region").size().sort_values(ascending=False).plot.bar(figsize=(9, 4), color="steelblue")
plt.title("地域別 取引件数", fontsize=14, fontweight="bold")
ax.text(0, 1.02, "(件)", transform=ax.transAxes, ha="left", fontsize=10)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

# 3. pivot_region_cat を Set2 で
ax = pivot_region_cat.plot.bar(stacked=True, figsize=(10, 5), colormap="Set2")
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f"{x/10000:,.0f}"))
plt.title("地域 × カテゴリ 売上内訳", fontsize=14, fontweight="bold")
ax.text(0, 1.02, "(万円)", transform=ax.transAxes, ha="left", fontsize=10)
plt.xticks(rotation=0)
plt.legend(loc="upper left", bbox_to_anchor=(1, 1))
plt.tight_layout()
plt.savefig(f"{CHARTS}/region_category_set2.png", dpi=150, bbox_inches="tight")
plt.show()
```

</details>

## よくあるエラー

### 1. グラフが Excel に貼り付けたら **小さくて荒い**
→ `dpi=150` 以上にする。資料用なら `dpi=200` でもOK。ファイルサイズと相談。

### 2. グラフが何枚か描いたあと、変なところに線が描き足される
→ `plt.show()` を呼ばずに次の描画を始めると、前のグラフに重なる。**各ブロックの最後に必ず `plt.show()`**。

### 3. `savefig` で日本語が **□□□** になる
→ `import japanize_matplotlib` が抜けているか、import するセルを実行していない。

### 4. 横棒の順番が **下から大きい順** になる
→ `Series[::-1]` で反転してから `barh` に渡す。または `barh` 後に `plt.gca().invert_yaxis()`。

### 5. 凡例がグラフに被って読めない
→ `plt.legend(loc="upper left", bbox_to_anchor=(1, 1))` で外に出す (3-4 で覚えたやつ)。

## まとめ

- グラフのコードは **すべて 3 章で習った型** をそのまま流用 (`Series.plot` / `pivot.plot.bar(stacked=True)` / `savefig`)
- 体裁の作法も 3-4 のまま (`japanize_matplotlib` / 単位は軸の上 / 桁区切り / tight_layout)
- 5 枚の PNG が `output/charts/` に揃ったので、次の 4-5 で **Excel レポートに貼り込み** ます

ここまでで「データ読込 → 結合 → 集計 → グラフ → 保存」の流れが揃いました。次の **4-5** では、集計結果と画像を **1 つの Excel ファイル** にまとめ、上司に送れる形にします。